# Template design audit

This notebook is used to inspect and audit the task-template layer before or alongside full dataset generation.

It focuses on:
- task template families
- instruction paraphrase pools
- reusable value pools
- sample rendered candidate examples
- quick checks for diversity and difficulty
- qualitative inspection of richer IE and QA templates

This notebook is not the main dataset-build notebook.
It is a design and auditing notebook for the template layer.

In [1]:
from pathlib import Path
import json
import sys
from collections import Counter, defaultdict

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions')

## 1) Imports

In [3]:
from src.templates import (
    SINGLE_LABEL_SET,
    MULTI_LABEL_SET,
    IE_SCHEMA_KEYS,
    RULE_SET,
    people,
    locations,
    dates,
    products,
    companies,
    other_locations,
    other_people,
    other_companies,
    other_dates,
    single_label_instructions,
    multi_label_instructions,
    ie_instructions,
    rule_instructions,
    qa_instructions,
    single_label_templates,
    single_label_value_map,
    multi_label_templates,
    topic_pool,
    multi_label_combinations,
    ie_templates,
    transformation_input_templates,
    qa_templates,
)

from src.generation import (
    generate_single_label_candidates,
    generate_multi_label_candidates,
    generate_ie_candidates,
    generate_transformation_candidates,
    generate_qa_candidates,
    generate_all_candidates,
    candidate_to_review_row,
    render_input_for_review,
    auto_flag_candidate,
    find_exact_duplicate_inputs,
)

## 2) Label sets, schemas, and rule inventory

In [4]:
print("SINGLE_LABEL_SET =", SINGLE_LABEL_SET)
print("MULTI_LABEL_SET  =", MULTI_LABEL_SET)
print("IE_SCHEMA_KEYS   =", IE_SCHEMA_KEYS)
print("RULE_SET         =", RULE_SET)

SINGLE_LABEL_SET = ['positive', 'negative', 'neutral']
MULTI_LABEL_SET  = ['finance', 'health', 'politics', 'sports', 'tech']
IE_SCHEMA_KEYS   = ['person', 'date', 'location']
RULE_SET         = ['lowercase', 'remove_punctuation', 'replace_numbers_with_NUM', 'remove_words_longer_than_6']


## 3) Core value pools

In [5]:
pool_summary = {
    "people": len(people),
    "locations": len(locations),
    "dates": len(dates),
    "products": len(products),
    "companies": len(companies),
    "other_locations": len(other_locations),
    "other_people": len(other_people),
    "other_companies": len(other_companies),
    "other_dates": len(other_dates),
}

print(json.dumps(pool_summary, indent=2, ensure_ascii=False))

{
  "people": 10,
  "locations": 10,
  "dates": 10,
  "products": 8,
  "companies": 8,
  "other_locations": 10,
  "other_people": 10,
  "other_companies": 6,
  "other_dates": 10
}


In [6]:
print("people:", people)
print()
print("locations:", locations)
print()
print("dates:", dates)

people: ['Alice Smith', 'Bruno Rossi', 'Carla Gomez', 'David Lee', 'Elena Marino', 'Farah Khan', 'Giulia Conti', 'Hassan Ali', 'Irene Novak', 'Jonas Weber']

locations: ['Rome', 'Milan', 'Paris', 'Berlin', 'Madrid', 'Lisbon', 'Vienna', 'Prague', 'Athens', 'Dublin']

dates: ['2024-01-15', '2024-03-21', '2024-06-10', '2024-09-05', '2025-02-14', '2025-04-30', '2025-07-12', '2025-10-03', '2026-01-08', '2026-02-19']


## 4) Instruction paraphrase pools

These are now intentionally diverse and should not be collapsed into a single instruction per task family.

In [7]:
instruction_pool_summary = {
    "single_label_instructions": len(single_label_instructions),
    "multi_label_instructions": len(multi_label_instructions),
    "ie_instructions": len(ie_instructions),
    "qa_instructions": len(qa_instructions),
    "rule_instruction_variants": {rule_name: len(pool) for rule_name, pool in rule_instructions.items()},
}

print(json.dumps(instruction_pool_summary, indent=2, ensure_ascii=False))

{
  "single_label_instructions": 6,
  "multi_label_instructions": 6,
  "ie_instructions": 6,
  "qa_instructions": 8,
  "rule_instruction_variants": {
    "lowercase": 4,
    "remove_punctuation": 4,
    "replace_numbers_with_NUM": 4,
    "remove_words_longer_than_6": 4
  }
}


In [8]:
print("Single-label instructions")
print(json.dumps(single_label_instructions, indent=2, ensure_ascii=False))
print()

print("Multi-label instructions")
print(json.dumps(multi_label_instructions, indent=2, ensure_ascii=False))
print()

print("IE instructions")
print(json.dumps(ie_instructions, indent=2, ensure_ascii=False))
print()

print("QA instructions")
print(json.dumps(qa_instructions, indent=2, ensure_ascii=False))

Single-label instructions
[
  "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "Return exactly one sentiment label from {positive, negative, neutral}.",
  "Choose one sentiment label only from {positive, negative, neutral}.",
  "Assign a single sentiment label from the allowed set {positive, negative, neutral}.",
  "Label the text with exactly one of {positive, negative, neutral}.",
  "Return one sentiment label only: {positive, negative, neutral}."
]

Multi-label instructions
[
  "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.",
  "Return every relevant topic label from {finance, health, politics, sports, tech}, sorted alphabetically.",
  "Choose all labels that apply from {finance, health, politics, sports, tech}. Return them in alphabetical order.",
  "Assign the applicable topic labels using only {finance, health, politics, sports, tech}. Sort the labels alphabetically

## 5) Rule instruction pools

In [9]:
print(json.dumps(rule_instructions, indent=2, ensure_ascii=False))

{
  "lowercase": [
    "Convert the text to lowercase.",
    "Rewrite the text in lowercase.",
    "Lowercase all letters in the text.",
    "Return the text with all alphabetic characters in lowercase."
  ],
  "remove_punctuation": [
    "Remove all punctuation from the text.",
    "Return the text with punctuation removed.",
    "Delete every punctuation mark from the text.",
    "Strip punctuation characters from the text."
  ],
  "replace_numbers_with_NUM": [
    "Replace every number in the text with <NUM>.",
    "Substitute each number in the text with <NUM>.",
    "Return the text with all numbers replaced by <NUM>.",
    "Replace every numeric sequence with <NUM>."
  ],
  "remove_words_longer_than_6": [
    "Remove all words longer than 6 characters from the text.",
    "Delete every word whose length is greater than 6 characters.",
    "Return the text after removing words longer than 6 characters.",
    "Drop all words with more than 6 characters."
  ]
}


## 6) Single-label template families

In [10]:
print("Number of single-label templates:", len(single_label_templates))
print(json.dumps(single_label_templates, indent=2, ensure_ascii=False))

Number of single-label templates: 6
[
  {
    "template_name": "product_review_balanced",
    "pattern": "I expected more from the {product}. It was {descriptor_1}, but the overall experience felt {descriptor_2}.",
    "instruction_pool": [
      "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
      "Return exactly one sentiment label from {positive, negative, neutral}.",
      "Choose one sentiment label only from {positive, negative, neutral}.",
      "Assign a single sentiment label from the allowed set {positive, negative, neutral}.",
      "Label the text with exactly one of {positive, negative, neutral}.",
      "Return one sentiment label only: {positive, negative, neutral}."
    ]
  },
  {
    "template_name": "service_feedback_natural",
    "pattern": "The staff were {descriptor_1} and the process was {descriptor_2}. I would describe the experience as {descriptor_3}.",
    "instruction_pool": [
      "Classify the sentiment of 

## 7) Single-label value map

In [11]:
print(json.dumps(single_label_value_map, indent=2, ensure_ascii=False))

{
  "positive": [
    {
      "descriptor_1": "smooth",
      "descriptor_2": "pleasant",
      "descriptor_3": "worth repeating"
    },
    {
      "descriptor_1": "reliable",
      "descriptor_2": "easy",
      "descriptor_3": "better than expected"
    },
    {
      "descriptor_1": "well organized",
      "descriptor_2": "helpful",
      "descriptor_3": "genuinely satisfying"
    },
    {
      "descriptor_1": "impressive",
      "descriptor_2": "straightforward",
      "descriptor_3": "very positive"
    }
  ],
  "negative": [
    {
      "descriptor_1": "confusing",
      "descriptor_2": "frustrating",
      "descriptor_3": "not worth repeating"
    },
    {
      "descriptor_1": "disappointing",
      "descriptor_2": "slow",
      "descriptor_3": "worse than expected"
    },
    {
      "descriptor_1": "poorly handled",
      "descriptor_2": "unhelpful",
      "descriptor_3": "quite negative"
    },
    {
      "descriptor_1": "rough",
      "descriptor_2": "annoying",
      "de

## 8) Multi-label template families

In [12]:
print("Number of multi-label templates:", len(multi_label_templates))
print(json.dumps(multi_label_templates, indent=2, ensure_ascii=False))

Number of multi-label templates: 8
[
  {
    "template_name": "institutional_announcement",
    "pattern": "The {actor} announced a new initiative involving {topic_item}. The statement also discussed implications for {secondary_item}.",
    "instruction_pool": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.",
      "Return every relevant topic label from {finance, health, politics, sports, tech}, sorted alphabetically.",
      "Choose all labels that apply from {finance, health, politics, sports, tech}. Return them in alphabetical order.",
      "Assign the applicable topic labels using only {finance, health, politics, sports, tech}. Sort the labels alphabetically.",
      "Return all matching topic labels from the allowed set {finance, health, politics, sports, tech}, in alphabetical order.",
      "Label the text with every applicable topic from {finance, health, politics, sports, tech}. Output the labels alphabe

## 9) Topic pool and label combinations

In [13]:
print("Topic pool")
print(json.dumps(topic_pool, indent=2, ensure_ascii=False))
print()
print("Multi-label combinations")
print(json.dumps(multi_label_combinations, indent=2, ensure_ascii=False))

Topic pool
{
  "politics": [
    "election reform",
    "ministerial decision",
    "government regulation",
    "parliament debate",
    "public policy",
    "municipal vote"
  ],
  "tech": [
    "AI system",
    "cloud platform",
    "mobile app",
    "data pipeline",
    "robotics tool",
    "software update",
    "machine learning model"
  ],
  "health": [
    "hospital policy",
    "vaccine program",
    "mental health service",
    "medical device",
    "nutrition study",
    "clinical software"
  ],
  "sports": [
    "football tournament",
    "tennis training app",
    "basketball analytics tool",
    "cycling event",
    "swimming program"
  ],
  "finance": [
    "bank loan",
    "tax policy",
    "stock market update",
    "budget plan",
    "investment fund",
    "procurement budget"
  ]
}

Multi-label combinations
[
  [
    "politics"
  ],
  [
    "tech"
  ],
  [
    "health"
  ],
  [
    "sports"
  ],
  [
    "finance"
  ],
  [
    "politics",
    "tech"
  ],
  [
    "tech

## 10) Information extraction templates

In [14]:
print("Number of IE templates:", len(ie_templates))
print(json.dumps(ie_templates, indent=2, ensure_ascii=False))

Number of IE templates: 6
[
  {
    "template_name": "meeting_announcement_rich",
    "pattern": "The agenda lists a remote check-in on {other_date}, but confirms that {person} will speak in {location} on {date}. A second venue was considered earlier and later removed.",
    "instruction_pool": [
      "Extract the fields person, date, and location from the text.",
      "Return the person, date, and location fields from the text.",
      "Identify and output person, date, and location from the text.",
      "Extract person, date, and location and return them as structured fields.",
      "Find the person, date, and location mentioned in the text.",
      "Return the values for person, date, and location from the text."
    ]
  },
  {
    "template_name": "conference_attendance_rich",
    "pattern": "On {date}, {person} attended a conference in {location}. Several attendees arrived the day before, and one side session was postponed to {other_date}.",
    "instruction_pool": [
      "Ex

## 11) Transformation templates

In [15]:
print("Number of transformation templates:", len(transformation_input_templates))
print(json.dumps(transformation_input_templates, indent=2, ensure_ascii=False))

Number of transformation templates: 7
[
  {
    "template_name": "mixed_case_statement",
    "pattern": "Today {person} Visited {location} With {n} Friends!"
  },
  {
    "template_name": "punctuated_note",
    "pattern": "Reminder: bring {n} tickets, {n2} pens, and a map to {location}."
  },
  {
    "template_name": "short_report",
    "pattern": "{person} bought {n} books and {n2} snacks in {location} yesterday."
  },
  {
    "template_name": "quote_and_symbol_note",
    "pattern": "\"{person}\" said: pack {n} cables, {n2} adapters, and re-check {location}!"
  },
  {
    "template_name": "slash_and_dash_update",
    "pattern": "Status update - {person}/{location}: {n} forms filed, {n2} pending."
  },
  {
    "template_name": "mixed_caps_checklist",
    "pattern": "CHECKLIST for {person}: Bring {n} IDs, {n2} copies, and call {location}."
  },
  {
    "template_name": "word_length_test",
    "pattern": "Tiny words vanish slowly when elephantine expressions dominate paragraphs."
  }
]


## 12) Extractive QA templates

These are especially important to inspect because the redesign made them richer and less trivial.

In [16]:
print("Number of QA templates:", len(qa_templates))
print(json.dumps(qa_templates, indent=2, ensure_ascii=False))

Number of QA templates: 12
[
  {
    "template_name": "schedule_passage_rich",
    "passage": "{person} left Rome on {other_date}, arrived in {location} on {date} for a training session, and later continued to {other_location} for a short meeting.",
    "question": "Where did {person} arrive on {date}?",
    "answer_field": "location",
    "instruction_pool": [
      "Answer the question using an exact span from the passage.",
      "Return the exact words from the passage that answer the question.",
      "Use a verbatim span from the passage.",
      "Copy the answer directly from the passage.",
      "Answer with the precise text appearing in the passage.",
      "Return only the exact answer span from the passage.",
      "Use the passage wording exactly for the answer.",
      "Extract the answer directly from the passage without paraphrasing."
    ],
    "required_fields": [
      "person",
      "location",
      "date",
      "other_date",
      "other_location"
    ]
  },
  {


## 13) Generate sample candidates by task

We generate only task-specific candidates here for inspection, not as the main dataset build step.

In [17]:
single_label_candidates = generate_single_label_candidates(start_id=0)
multi_label_candidates = generate_multi_label_candidates(start_id=0)
ie_candidates = generate_ie_candidates(start_id=0)
transformation_candidates = generate_transformation_candidates(start_id=0)
qa_candidates = generate_qa_candidates(start_id=0)

task_candidate_counts = {
    "single_label_classification": len(single_label_candidates),
    "multi_label_classification": len(multi_label_candidates),
    "information_extraction": len(ie_candidates),
    "rule_based_transformation": len(transformation_candidates),
    "extractive_qa": len(qa_candidates),
}

print(json.dumps(task_candidate_counts, indent=2, ensure_ascii=False))

{
  "single_label_classification": 50,
  "multi_label_classification": 50,
  "information_extraction": 50,
  "rule_based_transformation": 50,
  "extractive_qa": 50
}


## 14) Preview single-label candidates

In [18]:
for example in single_label_candidates[:5]:
    print("=" * 120)
    print(json.dumps(candidate_to_review_row(example), indent=2, ensure_ascii=False))

{
  "example_id": "slc_000",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Choose one sentiment label only from {positive, negative, neutral}.",
  "rendered_input": "The app looked rough, and most features worked, but using it felt annoying.",
  "gold_output": "{\"label\": \"negative\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "slc_001",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "rendered_input": "The app looked disappointing, and most features worked, but using it felt slow.",
  "gold_output": "{\"label\": \"negative\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "slc_002",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Return exactly one sentiment l

## 15) Preview multi-label candidates

In [19]:
for example in multi_label_candidates[:5]:
    print("=" * 120)
    print(json.dumps(candidate_to_review_row(example), indent=2, ensure_ascii=False))

{
  "example_id": "mlc_000",
  "task_name": "multi_label_classification",
  "template_name": "company_news",
  "instruction": "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.",
  "rendered_input": "Atlas Finance introduced a new effort around mental health service while explaining its connection to bank loan.",
  "gold_output": "{\"labels\": [\"finance\", \"health\"]}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "mlc_001",
  "task_name": "multi_label_classification",
  "template_name": "company_news",
  "instruction": "Assign the applicable topic labels using only {finance, health, politics, sports, tech}. Sort the labels alphabetically.",
  "rendered_input": "Meridian HealthWorks introduced a new effort around mental health service while explaining its connection to budget plan.",
  "gold_output": "{\"labels\": [\"finance\", \"health\"]}",
  "review_status": "pending",
  "review_note": ""
}
{
  "

## 16) Preview IE candidates

In [20]:
for example in ie_candidates[:5]:
    print("=" * 120)
    print(json.dumps(candidate_to_review_row(example), indent=2, ensure_ascii=False))

{
  "example_id": "ie_000",
  "task_name": "information_extraction",
  "template_name": "appointment_notice_rich",
  "instruction": "Extract person, date, and location and return them as structured fields.",
  "rendered_input": "Alice Smith was appointed in Berlin on 2024-03-21. Another internal memo mentioned a review meeting on 2024-09-14, but no change to the appointment record.",
  "gold_output": "{\"person\": \"Alice Smith\", \"date\": \"2024-03-21\", \"location\": \"Berlin\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "ie_001",
  "task_name": "information_extraction",
  "template_name": "appointment_notice_rich",
  "instruction": "Extract the fields person, date, and location from the text.",
  "rendered_input": "Alice Smith was appointed in Vienna on 2025-02-14. Another internal memo mentioned a review meeting on 2025-07-29, but no change to the appointment record.",
  "gold_output": "{\"person\": \"Alice Smith\", \"date\": \"2025-02-14\", \"location

## 17) Preview transformation candidates

In [21]:
for example in transformation_candidates[:5]:
    print("=" * 120)
    print(json.dumps(candidate_to_review_row(example), indent=2, ensure_ascii=False))

{
  "example_id": "rbt_000",
  "task_name": "rule_based_transformation",
  "template_name": "mixed_caps_checklist__lowercase",
  "instruction": "Convert the text to lowercase.",
  "rendered_input": "CHECKLIST for David Lee: Bring 3 IDs, 12 copies, and call Athens.",
  "gold_output": "{\"text\": \"checklist for david lee: bring 3 ids, 12 copies, and call athens.\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "rbt_001",
  "task_name": "rule_based_transformation",
  "template_name": "mixed_caps_checklist__lowercase",
  "instruction": "Convert the text to lowercase.",
  "rendered_input": "CHECKLIST for Hassan Ali: Bring 3 IDs, 12 copies, and call Paris.",
  "gold_output": "{\"text\": \"checklist for hassan ali: bring 3 ids, 12 copies, and call paris.\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "rbt_002",
  "task_name": "rule_based_transformation",
  "template_name": "mixed_caps_checklist__remove_punctuation",
  "instruction": "Retu

## 18) Preview QA candidates

In [22]:
for example in qa_candidates[:8]:
    print("=" * 120)
    print(json.dumps(candidate_to_review_row(example), indent=2, ensure_ascii=False))

{
  "example_id": "qa_000",
  "task_name": "extractive_qa",
  "template_name": "deployment_update_passage",
  "instruction": "Answer the question using an exact span from the passage.",
  "rendered_input": "PASSAGE: The deployment note says BlueRiver Health will roll out the medical software on 2025-10-03 after testing in Vienna. A separate memo about Vertex Retail Tech concerned a different tool.\nQUESTION: Which company will roll out the medical software on 2025-10-03?",
  "gold_output": "{\"answer\": \"BlueRiver Health\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "qa_001",
  "task_name": "extractive_qa",
  "template_name": "deployment_update_passage",
  "instruction": "Answer with the precise text appearing in the passage.",
  "rendered_input": "PASSAGE: The deployment note says BlueRiver Health will roll out the solar battery on 2024-09-05 after testing in Paris. A separate memo about BrightCore Systems concerned a different tool.\nQUESTION: Which comp

## 19) Inspect rendered inputs only

This is useful when you want to quickly check whether the task inputs look too repetitive or too easy.

In [23]:
for example in qa_candidates[:5]:
    print("=" * 120)
    print("example_id:", example.example_id)
    print("template_name:", example.template_name)
    print("instruction:", example.instruction)
    print("-" * 120)
    print(render_input_for_review(example))
    print()

example_id: qa_000
template_name: deployment_update_passage
instruction: Answer the question using an exact span from the passage.
------------------------------------------------------------------------------------------------------------------------
PASSAGE: The deployment note says BlueRiver Health will roll out the medical software on 2025-10-03 after testing in Vienna. A separate memo about Vertex Retail Tech concerned a different tool.
QUESTION: Which company will roll out the medical software on 2025-10-03?

example_id: qa_001
template_name: deployment_update_passage
instruction: Answer with the precise text appearing in the passage.
------------------------------------------------------------------------------------------------------------------------
PASSAGE: The deployment note says BlueRiver Health will roll out the solar battery on 2024-09-05 after testing in Paris. A separate memo about BrightCore Systems concerned a different tool.
QUESTION: Which company will roll out th

## 20) Inspect instruction diversity in sampled candidates

In [24]:
all_candidates = generate_all_candidates()

instructions_by_task = defaultdict(set)
for example in all_candidates:
    instructions_by_task[example.task_name].add(example.instruction)

instruction_diversity_summary = {
    task_name: {
        "num_unique_instructions": len(instructions),
        "instructions": sorted(instructions),
    }
    for task_name, instructions in instructions_by_task.items()
}

print(json.dumps(instruction_diversity_summary, indent=2, ensure_ascii=False))

{
  "single_label_classification": {
    "num_unique_instructions": 6,
    "instructions": [
      "Assign a single sentiment label from the allowed set {positive, negative, neutral}.",
      "Choose one sentiment label only from {positive, negative, neutral}.",
      "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
      "Label the text with exactly one of {positive, negative, neutral}.",
      "Return exactly one sentiment label from {positive, negative, neutral}.",
      "Return one sentiment label only: {positive, negative, neutral}."
    ]
  },
  "multi_label_classification": {
    "num_unique_instructions": 6,
    "instructions": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.",
      "Assign the applicable topic labels using only {finance, health, politics, sports, tech}. Sort the labels alphabetically.",
      "Choose all labels that apply from {finance, heal

## 21) Count template usage in sampled candidates


In [25]:
template_counts_by_task = defaultdict(Counter)

for example in all_candidates:
    template_counts_by_task[example.task_name][example.template_name] += 1

template_counts_by_task = {
    task_name: dict(counter)
    for task_name, counter in template_counts_by_task.items()
}

print(json.dumps(template_counts_by_task, indent=2, ensure_ascii=False))

{
  "single_label_classification": {
    "app_experience_note": 9,
    "delivery_comment_mixed": 9,
    "event_reaction_short_review": 8,
    "product_review_balanced": 8,
    "service_feedback_natural": 8,
    "store_visit_note": 8
  },
  "multi_label_classification": {
    "company_news": 7,
    "event_report": 7,
    "institutional_announcement": 6,
    "newsletter_excerpt": 6,
    "panel_discussion_note": 6,
    "policy_summary": 6,
    "press_briefing": 6,
    "project_update": 6
  },
  "information_extraction": {
    "appointment_notice_rich": 9,
    "conference_attendance_rich": 9,
    "event_registration_rich": 8,
    "meeting_announcement_rich": 8,
    "speaker_log_entry": 8,
    "travel_schedule_notice": 8
  },
  "rule_based_transformation": {
    "mixed_caps_checklist__lowercase": 3,
    "mixed_caps_checklist__remove_punctuation": 2,
    "mixed_caps_checklist__remove_words_longer_than_6": 2,
    "mixed_caps_checklist__replace_numbers_with_NUM": 2,
    "mixed_case_statement__

## 22) Auto-flag generated candidates

This is only a warning-oriented inspection step.

In [26]:
flagged = []

for example in all_candidates:
    issues = auto_flag_candidate(example)
    if issues:
        flagged.append({
            "example_id": example.example_id,
            "task_name": example.task_name,
            "template_name": example.template_name,
            "issues": issues,
        })

len(flagged)

0

In [27]:
for row in flagged[:15]:
    print(row)

## 23) Check for exact duplicate rendered inputs

In [28]:
duplicate_inputs = find_exact_duplicate_inputs(all_candidates)
print("Number of duplicate rendered-input groups:", len(duplicate_inputs))

Number of duplicate rendered-input groups: 0


In [29]:
if duplicate_inputs:
    for rendered, ids in list(duplicate_inputs.items())[:5]:
        print("=" * 120)
        print("IDs:", ids)
        print(rendered)
else:
    print("No exact duplicate rendered inputs found.")

No exact duplicate rendered inputs found.


## 24) Quick QA audit

We explicitly inspect answer uniqueness and passage richness.

In [30]:
qa_audit_rows = []

for example in qa_candidates[:15]:
    passage = example.input_data["passage"]
    answer = example.gold_output["answer"]
    qa_audit_rows.append({
        "example_id": example.example_id,
        "template_name": example.template_name,
        "answer": answer,
        "answer_count_in_passage": passage.count(answer),
        "passage_length": len(passage),
        "question_length": len(example.input_data["question"]),
    })

print(json.dumps(qa_audit_rows, indent=2, ensure_ascii=False))

[
  {
    "example_id": "qa_000",
    "template_name": "deployment_update_passage",
    "answer": "BlueRiver Health",
    "answer_count_in_passage": 1,
    "passage_length": 184,
    "question_length": 63
  },
  {
    "example_id": "qa_001",
    "template_name": "deployment_update_passage",
    "answer": "BlueRiver Health",
    "answer_count_in_passage": 1,
    "passage_length": 180,
    "question_length": 60
  },
  {
    "example_id": "qa_002",
    "template_name": "deployment_update_passage",
    "answer": "BlueRiver Health",
    "answer_count_in_passage": 1,
    "passage_length": 180,
    "question_length": 60
  },
  {
    "example_id": "qa_003",
    "template_name": "deployment_update_passage",
    "answer": "BlueRiver Health",
    "answer_count_in_passage": 1,
    "passage_length": 188,
    "question_length": 64
  },
  {
    "example_id": "qa_004",
    "template_name": "deployment_update_passage",
    "answer": "BlueRiver Health",
    "answer_count_in_passage": 1,
    "passage_len

## 25) Quick IE audit

We inspect whether richer IE examples still keep the gold triplet readable and plausible.

In [31]:
for example in ie_candidates[:8]:
    print("=" * 120)
    print("example_id:", example.example_id)
    print("template_name:", example.template_name)
    print("gold_output:", example.gold_output)
    print("-" * 120)
    print(example.input_data["text"])
    print()

example_id: ie_000
template_name: appointment_notice_rich
gold_output: {'person': 'Alice Smith', 'date': '2024-03-21', 'location': 'Berlin'}
------------------------------------------------------------------------------------------------------------------------
Alice Smith was appointed in Berlin on 2024-03-21. Another internal memo mentioned a review meeting on 2024-09-14, but no change to the appointment record.

example_id: ie_001
template_name: appointment_notice_rich
gold_output: {'person': 'Alice Smith', 'date': '2025-02-14', 'location': 'Vienna'}
------------------------------------------------------------------------------------------------------------------------
Alice Smith was appointed in Vienna on 2025-02-14. Another internal memo mentioned a review meeting on 2025-07-29, but no change to the appointment record.

example_id: ie_002
template_name: appointment_notice_rich
gold_output: {'person': 'Alice Smith', 'date': '2024-06-10', 'location': 'Madrid'}
---------------------

## 26) Quick difficulty audit notes

This is a place to add handwritten qualitative observations while reviewing the generated samples.

In [32]:
difficulty_audit_template = {
    "single_label_classification": [
        "Are the sentiment cues too obvious or reasonably natural?",
        "Do neutral cases look genuinely neutral rather than weakly positive/negative?",
    ],
    "multi_label_classification": [
        "Do label combinations feel realistic?",
        "Are examples too directly lexicalized by the label words?",
    ],
    "information_extraction": [
        "Do distractor dates/locations make the examples richer without becoming ambiguous?",
        "Are target fields still uniquely recoverable?",
    ],
    "rule_based_transformation": [
        "Do inputs contain enough punctuation/casing/number variety?",
        "Are the rules still fully deterministic and easy to score automatically?",
    ],
    "extractive_qa": [
        "Do passages contain multiple candidate spans often enough?",
        "Are answer spans still unique?",
        "Do question forms vary enough?",
    ],
}

print(json.dumps(difficulty_audit_template, indent=2, ensure_ascii=False))

{
  "single_label_classification": [
    "Are the sentiment cues too obvious or reasonably natural?",
    "Do neutral cases look genuinely neutral rather than weakly positive/negative?"
  ],
  "multi_label_classification": [
    "Do label combinations feel realistic?",
    "Are examples too directly lexicalized by the label words?"
  ],
  "information_extraction": [
    "Do distractor dates/locations make the examples richer without becoming ambiguous?",
    "Are target fields still uniquely recoverable?"
  ],
  "rule_based_transformation": [
    "Do inputs contain enough punctuation/casing/number variety?",
    "Are the rules still fully deterministic and easy to score automatically?"
  ],
  "extractive_qa": [
    "Do passages contain multiple candidate spans often enough?",
    "Are answer spans still unique?",
    "Do question forms vary enough?"
  ]
}


## 27) Full candidate generation snapshot

This is not a replacement for the main dataset build notebook, but it lets us inspect the whole candidate pool at the template-design level.

In [33]:
print("Total generated candidates:", len(all_candidates))

task_counts = Counter(example.task_name for example in all_candidates)
print(json.dumps(task_counts, indent=2, ensure_ascii=False))

Total generated candidates: 250
{
  "single_label_classification": 50,
  "multi_label_classification": 50,
  "information_extraction": 50,
  "rule_based_transformation": 50,
  "extractive_qa": 50
}


In [34]:
for example in all_candidates[:10]:
    print("=" * 120)
    print("example_id:", example.example_id)
    print("task_name:", example.task_name)
    print("template_name:", example.template_name)
    print("instruction:", example.instruction)
    print("gold_output:", example.gold_output)
    print("-" * 120)
    print(render_input_for_review(example))
    print()

example_id: slc_000
task_name: single_label_classification
template_name: app_experience_note
instruction: Choose one sentiment label only from {positive, negative, neutral}.
gold_output: {'label': 'negative'}
------------------------------------------------------------------------------------------------------------------------
The app looked rough, and most features worked, but using it felt annoying.

example_id: slc_001
task_name: single_label_classification
template_name: app_experience_note
instruction: Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.
gold_output: {'label': 'negative'}
------------------------------------------------------------------------------------------------------------------------
The app looked disappointing, and most features worked, but using it felt slow.

example_id: slc_002
task_name: single_label_classification
template_name: app_experience_note
instruction: Return exactly one sentiment label from {posit

## 28) Template-layer sanity checks

In [35]:
assert len(single_label_templates) >= 4, "Expected multiple single-label templates"
assert len(multi_label_templates) >= 4, "Expected multiple multi-label templates"
assert len(ie_templates) >= 4, "Expected multiple IE templates"
assert len(transformation_input_templates) >= 4, "Expected multiple transformation templates"
assert len(qa_templates) >= 8, "Expected richer QA template coverage"

assert len(single_label_instructions) >= 4, "Expected paraphrase diversity for single-label instructions"
assert len(multi_label_instructions) >= 4, "Expected paraphrase diversity for multi-label instructions"
assert len(ie_instructions) >= 4, "Expected paraphrase diversity for IE instructions"
assert len(qa_instructions) >= 4, "Expected paraphrase diversity for QA instructions"

print("Template-design sanity checks passed.")

Template-design sanity checks passed.


## 29) Compact design summary

In [36]:
design_summary = {
    "num_single_label_templates": len(single_label_templates),
    "num_multi_label_templates": len(multi_label_templates),
    "num_ie_templates": len(ie_templates),
    "num_transformation_templates": len(transformation_input_templates),
    "num_qa_templates": len(qa_templates),
    "num_single_label_instruction_variants": len(single_label_instructions),
    "num_multi_label_instruction_variants": len(multi_label_instructions),
    "num_ie_instruction_variants": len(ie_instructions),
    "num_qa_instruction_variants": len(qa_instructions),
    "rule_instruction_variants": {rule: len(pool) for rule, pool in rule_instructions.items()},
    "generated_candidates_total": len(all_candidates),
}

print(json.dumps(design_summary, indent=2, ensure_ascii=False))

{
  "num_single_label_templates": 6,
  "num_multi_label_templates": 8,
  "num_ie_templates": 6,
  "num_transformation_templates": 7,
  "num_qa_templates": 12,
  "num_single_label_instruction_variants": 6,
  "num_multi_label_instruction_variants": 6,
  "num_ie_instruction_variants": 6,
  "num_qa_instruction_variants": 8,
  "rule_instruction_variants": {
    "lowercase": 4,
    "remove_punctuation": 4,
    "replace_numbers_with_NUM": 4,
    "remove_words_longer_than_6": 4
  },
  "generated_candidates_total": 250
}
